## Notebook to query Uniprot for PDB-ligand

Create Uniprot-PDB-ligand data table query by Uniprot_ID

Input:
- Any list of uniprot_ids

In [10]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

repo_root = !git rev-parse --show-toplevel
repo_root = repo_root[0]
os.chdir(repo_root)

data_dir = Path("data") / "human_reference_proteome"
os.makedirs(data_dir, exist_ok=True)
os.makedirs(data_dir / "input", exist_ok=True)
os.makedirs(data_dir / "working", exist_ok=True)

uniprot_cache_dir = Path("data/uniprot_api_cache/uniprotkb")

ligand_properties_csv = data_dir / "working" / "ligand_properties.csv"

# Hardcoded defaults for reruns and downstream notebook cells
input_txt = data_dir / "input" / "human_reference_proteome_uniprot_ids.txt"
raw_out_csv = data_dir / "human_reference_proteome_pdbs.csv"
ligand_out_csv = data_dir / "human_reference_proteome_pdb_ligands.csv"
frag_out_csv = data_dir / "human_reference_proteome_drug_like_fragments.csv"
blacklist_txt = Path("blacklist.txt")
slurm_script = Path("scripts/slurm/uniprot_to_pdb_ligand.sh")

___
### Step 1 - Query Uniprot and PDB API for df_all
Run slurm job to query all human uniprot_ids in the PDB

In [2]:
import requests

# Get all UniProt IDs for the human reference proteome ~ 20k canonical reviewed sequences
url = (
    "https://rest.uniprot.org/uniprotkb/stream"
    "?query=proteome:UP000005640"
    "+AND+reviewed:true"
    "+AND+is_isoform:false"
    "&fields=accession"
    "&format=tsv"
)

r = requests.get(url)
r.raise_for_status()

# Skip header line ("Entry")
uniprot_ids = r.text.strip().splitlines()[1:]

print(f"Retrieved {len(uniprot_ids)} UniProt accessions")
print(uniprot_ids[:10])

Retrieved 20416 UniProt accessions
['A0A087X1C5', 'A0A096LP01', 'A0A0B4J2F0', 'A0A0C5B5G6', 'A0A0K2S4Q6', 'A0A0U1RRE5', 'A0A1B0GTW7', 'A0A2R8Y7D0', 'A0A8I5KQE6', 'A0AV02']


In [3]:
# Write to input_txt file
with open(input_txt, "w") as f:
    for id in uniprot_ids:
        f.write(id + "\n")
print(f"Prepared {len(uniprot_ids)} unique UniProt IDs in {input_txt}")

Prepared 20416 unique UniProt IDs in data/human_reference_proteome/input/human_reference_proteome_uniprot_ids.txt


In [ ]:
import subprocess

submit_cmd = [
    "sbatch",
    str(slurm_script),
    str(input_txt),
    str(raw_out_csv),
    str(blacklist_txt),
]
result = subprocess.run(submit_cmd, check=True, capture_output=True, text=True)
print(result.stdout.strip())

print("Monitor logs: logs/uniprot_ligands-<jobid>.out")


Submitted batch job 57104984
Monitor logs: logs/uniprot_ligands-<jobid>.out


___
### Step 2 - Enrich df_all with chemical information
After slurm job completes, read `raw_out_csv` as `df_all` (All PDB of human uniprot regardless whether a ligand is present)

In [4]:
import pandas as pd
from pathlib import Path

if not raw_out_csv.exists():
    raise FileNotFoundError(
        f"{raw_out_csv} not found yet. Wait for SLURM job completion, then rerun this cell."
    )

df_all = pd.read_csv(raw_out_csv)
print(f"Loaded {len(df_all)} rows from {raw_out_csv}")
df_all.head()

Loaded 126025 rows from data/human_reference_proteome/human_reference_proteome_pdbs.csv


,uniprot_id,gene_name,recommendedName,pdb_id,protein_chain,ligand_chain,ligand_code,ligand_name,smiles,percent_intracellular,formula,mw,qed,num_carbon,num_N_O,uniprot_id_count
0,A0AV96,RBM47,RNA-binding protein 47,2DIS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A0JLT2,MED19,Mediator of RNA polymerase II transcription su...,7EMF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A0JLT2,MED19,Mediator of RNA polymerase II transcription su...,7ENA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A0JLT2,MED19,Mediator of RNA polymerase II transcription su...,7ENC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A0JLT2,MED19,Mediator of RNA polymerase II transcription su...,7ENJ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Get gene_name and recommendedName from uniprot_cache_dir/<uniprot_id>.json
import json


def get_uniprot_metadata(uniprot_id):
    """Return (gene_name, recommendedName) from cached UniProt JSON."""
    uniprot_json_path = uniprot_cache_dir / f"{uniprot_id}.json"
    if not uniprot_json_path.is_file():
        return None, None

    with open(uniprot_json_path, "r", encoding="utf-8") as f:
        envelope = json.load(f)

    if envelope.get("status_code") != 200:
        return None, None

    data = envelope.get("body") or {}

    gene_name = None
    genes = data.get("genes") or []
    if genes:
        gene_name = (genes[0].get("geneName") or {}).get("value")

    recommended_name = None
    protein_desc = data.get("proteinDescription") or {}
    rec = protein_desc.get("recommendedName") or {}
    recommended_name = (rec.get("fullName") or {}).get("value")

    if recommended_name is None:
        submission_names = protein_desc.get("submissionNames") or []
        if submission_names:
            recommended_name = (submission_names[0].get("fullName") or {}).get("value")

    return gene_name, recommended_name


unique_uniprot_ids = df_all["uniprot_id"].dropna().unique()
df_metadata = pd.DataFrame(
    [get_uniprot_metadata(uid) for uid in unique_uniprot_ids],
    columns=["gene_name", "recommendedName"],
    index=unique_uniprot_ids,
).reset_index().rename(columns={"index": "uniprot_id"})

df_all = df_all.drop(columns=["gene_name", "recommendedName"], errors="ignore")
df_all = df_all.merge(df_metadata, on="uniprot_id", how="left")

# Reorder columns to have "uniprot_id", "gene_name", "recommendedName" first, rest in existing order
first_cols = ["uniprot_id", "gene_name", "recommendedName"]
other_cols = [c for c in df_all.columns if c not in first_cols]
df_all = df_all[first_cols + other_cols]

df_all.to_csv(raw_out_csv, index=False)
df_all.head()


In [ ]:
# Function to convert smiles to molecular formula and molecular weight using RDKit
from rdkit import Chem

def smiles_to_formula(smiles):
    """Convert SMILES to molecular formula using RDKit's rdMolDescriptors."""
    from rdkit.Chem import rdMolDescriptors
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return rdMolDescriptors.CalcMolFormula(mol)

def smiles_to_mw(smiles):
    """Convert SMILES to molecular weight using RDKit's rdMolDescriptors."""
    from rdkit.Chem import rdMolDescriptors
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return rdMolDescriptors.CalcExactMolWt(mol)

def smiles_to_qed(smiles):
    """Compute QED score using RDKit's QED module."""
    from rdkit.Chem import QED
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return QED.qed(mol)

def smiles_to_num_carbon(smiles):
    """Count the number of carbon atoms in a molecule."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return sum(1 for atom in mol.GetAtoms() if atom.GetSymbol() == 'C')

def smiles_to_num_N_O(smiles):
    """Count the number of N and O atoms in a molecule."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return sum(1 for atom in mol.GetAtoms() if atom.GetSymbol() in ['N', 'O'])


smiles = "O=C1NC(=O)CCC1N3C(=O)c2cccc(c2C3=O)N" # pomalidomide
print("Formula:", smiles_to_formula(smiles))
print("Exact MW:", smiles_to_mw(smiles))
print("QED:", smiles_to_qed(smiles))
print("Num Carbon:", smiles_to_num_carbon(smiles))
print("Num N and O:", smiles_to_num_N_O(smiles))

Formula: C13H11N3O4
Exact MW: 273.07495583200006
QED: 0.5375391722332333
Num Carbon: 13
Num N and O: 7


In [11]:
# Get a df of unique non-null ligands
df_unique_ligands = df_all[["ligand_code", "smiles"]]
df_unique_ligands = df_unique_ligands[df_unique_ligands["smiles"].notna()].drop_duplicates(subset="smiles")
print(df_unique_ligands.shape)
df_unique_ligands.head()

(21879, 2)


,ligand_code,smiles
12,OFN,CCCCCCCCCCCCCCCCCC(=O)CC(=O)SCCNC(=O)CCNC(=O)[...
13,37X,CCCCCCC(CCCCCC)(CO[C@@H]1[C@H]([C@@H]([C@H]([C...
17,EGC,CC(C)(C)CC(C)(C)c1ccc(cc1)OCCOCCOCCOCCOCCOCCOC...
18,PEE,CCCCCCCC/C=C\CCCCCCCC(=O)OC[C@H](COP(=O)(O)OCC...
26,GNP,c1nc2c(n1[C@H]3[C@@H]([C@@H]([C@H](O3)CO[P@](=...


In [12]:
# Compute properties for df_unique_ligands row-wise with progress bar
from tqdm.auto import tqdm

def valid_smiles(s):
    return isinstance(s, str) and s.strip() != ""

def compute_ligand_props(row):
    s = row["smiles"]
    if not valid_smiles(s):
        return {
            "formula": None,
            "mw": None,
            "qed": None,
            "num_carbon": None,
            "num_N_O": None,
        }
    return {
        "formula": smiles_to_formula(s),
        "mw": smiles_to_mw(s),
        "qed": smiles_to_qed(s),
        "num_carbon": smiles_to_num_carbon(s),
        "num_N_O": smiles_to_num_N_O(s),
    }

ligand_props = []
for _, row in tqdm(df_unique_ligands.iterrows(), total=len(df_unique_ligands), desc="Processing ligands"):
    ligand_props.append(compute_ligand_props(row))
ligand_props_df = pd.DataFrame(ligand_props)

# Concat properties with df_unique_ligands
df_unique_ligands = pd.concat([df_unique_ligands.reset_index(drop=True), ligand_props_df], axis=1)

df_unique_ligands.to_csv(ligand_properties_csv, index=False)
df_unique_ligands


Processing ligands:   0%|          | 0/21879 [00:00<?, ?it/s]

[12:55:18] Explicit valence for atom # 0 Be, 4, is greater than permitted
[12:55:18] Explicit valence for atom # 0 Be, 4, is greater than permitted
[12:55:18] Explicit valence for atom # 0 Be, 4, is greater than permitted
[12:55:18] Explicit valence for atom # 0 Be, 4, is greater than permitted
[12:55:18] Explicit valence for atom # 0 Be, 4, is greater than permitted
[12:55:19] Explicit valence for atom # 24 N, 4, is greater than permitted
[12:55:19] Explicit valence for atom # 24 N, 4, is greater than permitted
[12:55:19] Explicit valence for atom # 24 N, 4, is greater than permitted
[12:55:19] Explicit valence for atom # 24 N, 4, is greater than permitted
[12:55:19] Explicit valence for atom # 24 N, 4, is greater than permitted
[12:55:19] Explicit valence for atom # 25 N, 4, is greater than permitted
[12:55:19] Explicit valence for atom # 25 N, 4, is greater than permitted
[12:55:19] Explicit valence for atom # 25 N, 4, is greater than permitted
[12:55:19] Explicit valence for atom #

,ligand_code,smiles,formula,mw,qed,num_carbon,num_N_O
0,OFN,CCCCCCCCCCCCCCCCCC(=O)CC(=O)SCCNC(=O)CCNC(=O)[...,C41H72N7O18P3S,1075.386739,0.024703,41.0,25.0
1,37X,CCCCCCC(CCCCCC)(CO[C@@H]1[C@H]([C@@H]([C@H]([C...,C27H52O12,568.345877,0.099222,27.0,12.0
2,EGC,CC(C)(C)CC(C)(C)c1ccc(cc1)OCCOCCOCCOCCOCCOCCOC...,C32H58O10,602.402998,0.135862,32.0,10.0
3,PEE,CCCCCCCC/C=C\CCCCCCCC(=O)OC[C@H](COP(=O)(O)OCC...,C41H78NO8P,743.546505,0.027285,41.0,9.0
4,GNP,c1nc2c(n1[C@H]3[C@@H]([C@@H]([C@H](O3)CO[P@](=...,C10H17N6O13P3,522.006644,0.158512,10.0,19.0
...,...,...,...,...,...,...,...
21874,KW0,CCCCCCCC/C=C\CCCCCCCC(=O)OC[C@@H](COCC)OP(=O)(...,C26H50NO9P,551.322319,0.054161,26.0,10.0
21875,KW3,CCOC[C@H](COC(=O)CCc1ccccc1OCc2cccc(c2)Oc3cccc...,C30H36NO11P,617.202598,0.127309,30.0,12.0
21876,EN6,Cc1c(c(n(n1)c2cccc3c2sc(c3)Cc4cccc(c4)C(F)(F)F...,C24H22F3N3O3S,489.133397,0.362885,24.0,6.0
21877,5NG,CC(C)(COP(=O)(O)OP(=O)(O)OC[C@@H]1[C@H]([C@H](...,C42H70N14O32P6S2,1532.214768,0.012192,42.0,46.0


In [13]:
# Merge df_unique_ligands with df_all on both ligand_code and smiles
df_unique_ligands = pd.read_csv(ligand_properties_csv)
df_all = pd.merge(df_all, df_unique_ligands, on=["ligand_code", "smiles"], how="left")
df_all.head()

,uniprot_id,gene_name,recommendedName,pdb_id,protein_chain,ligand_chain,ligand_code,ligand_name,smiles,percent_intracellular,formula,mw,qed,num_carbon,num_N_O
0,A0AV96,RBM47,RNA-binding protein 47,2DIS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A0JLT2,MED19,Mediator of RNA polymerase II transcription su...,7EMF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A0JLT2,MED19,Mediator of RNA polymerase II transcription su...,7ENA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A0JLT2,MED19,Mediator of RNA polymerase II transcription su...,7ENC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A0JLT2,MED19,Mediator of RNA polymerase II transcription su...,7ENJ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# For each unique ligand_code in df_all, check how many uniprot_id are associated with it
df_ligand_uniprot_count = df_all.groupby("ligand_code")["uniprot_id"].nunique().reset_index(name="uniprot_id_count")

# Add uniprot_id_count to df_all
df_all["uniprot_id_count"] = df_all["ligand_code"].map(df_ligand_uniprot_count.set_index("ligand_code")["uniprot_id_count"])

df_all.to_csv(raw_out_csv, index=False)
df_all.head()

,uniprot_id,gene_name,recommendedName,pdb_id,protein_chain,ligand_chain,ligand_code,ligand_name,smiles,percent_intracellular,formula,mw,qed,num_carbon,num_N_O,uniprot_id_count
0,A0AV96,RBM47,RNA-binding protein 47,2DIS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A0JLT2,MED19,Mediator of RNA polymerase II transcription su...,7EMF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A0JLT2,MED19,Mediator of RNA polymerase II transcription su...,7ENA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A0JLT2,MED19,Mediator of RNA polymerase II transcription su...,7ENC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A0JLT2,MED19,Mediator of RNA polymerase II transcription su...,7ENJ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# Enrich df_all with "method" and "resolution" column from pdb_api_cache/<pdb_id>.json
import json

pdb_cache_dir = Path("data/pdb_api_cache/entry")


def load_pdb_entry_metadata(pdb_id: str) -> dict:
    cache_path = pdb_cache_dir / f"{pdb_id}.json"
    if not cache_path.is_file():
        return {"pdb_id": pdb_id, "method": None, "resolution": None}

    with cache_path.open(encoding="utf-8") as handle:
        body = json.load(handle).get("body") or {}

    exptl = body.get("exptl") or []
    method = exptl[0].get("method") if exptl else None

    info = body.get("rcsb_entry_info") or {}
    resolution = None
    res_combined = info.get("resolution_combined")
    if res_combined:
        resolution = res_combined[0]
    else:
        res_high = info.get("diffrn_resolution_high")
        if isinstance(res_high, dict):
            resolution = res_high.get("value")

    return {"pdb_id": pdb_id, "method": method, "resolution": resolution}


unique_pdb_ids = df_all["pdb_id"].dropna().unique()
df_pdb_metadata = pd.DataFrame(
    load_pdb_entry_metadata(pdb_id) for pdb_id in unique_pdb_ids
)

df_all = df_all.drop(columns=["method", "resolution"], errors="ignore")
df_all = df_all.merge(df_pdb_metadata, on="pdb_id", how="left")

df_all.to_csv(raw_out_csv, index=False)
df_all.head()

,uniprot_id,gene_name,recommendedName,pdb_id,protein_chain,ligand_chain,ligand_code,ligand_name,smiles,percent_intracellular,formula,mw,qed,num_carbon,num_N_O,uniprot_id_count,method,resolution
0,A0AV96,RBM47,RNA-binding protein 47,2DIS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SOLUTION NMR,NaN
1,A0JLT2,MED19,Mediator of RNA polymerase II transcription su...,7EMF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ELECTRON MICROSCOPY,3.50
2,A0JLT2,MED19,Mediator of RNA polymerase II transcription su...,7ENA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ELECTRON MICROSCOPY,4.07
3,A0JLT2,MED19,Mediator of RNA polymerase II transcription su...,7ENC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ELECTRON MICROSCOPY,4.13
4,A0JLT2,MED19,Mediator of RNA polymerase II transcription su...,7ENJ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ELECTRON MICROSCOPY,4.40


___
### Step 3 - Filter df_all to entries with valid ligands

In [9]:
# Filter df_all to only rows with a non-empty smiles
df_pdb_ligands = df_all[df_all["smiles"].notna()]

# Filter to include only specific ligands
df_pdb_ligands = df_pdb_ligands[df_pdb_ligands["uniprot_id_count"] <= 3]

# Remove ligands with no carbon atoms
df_pdb_ligands = df_pdb_ligands[df_pdb_ligands["num_carbon"] > 0]

# Remove ligands with mw < 120
df_pdb_ligands = df_pdb_ligands[df_pdb_ligands["mw"] > 120]

print(f"df_pdb_ligands.shape: {df_pdb_ligands.shape}")
df_pdb_ligands.to_csv(ligand_out_csv, index=False)
df_pdb_ligands.head()


df_pdb_ligands.shape: (28153, 18)


,uniprot_id,gene_name,recommendedName,pdb_id,protein_chain,ligand_chain,ligand_code,ligand_name,smiles,percent_intracellular,formula,mw,qed,num_carbon,num_N_O,uniprot_id_count,method,resolution
12,A1L3X0,ELOVL7,Very long chain fatty acid elongase 7,6Y7F,A,A,OFN,"~{S}-[2-[3-[[(2~{R})-4-[[[(2~{R},3~{S},4~{R},5...",CCCCCCCCCCCCCCCCCC(=O)CC(=O)SCCNC(=O)CCNC(=O)[...,0.213523,C41H72N7O18P3S,1075.386739,0.024703,41.0,25.0,1.0,X-RAY DIFFRACTION,2.052
13,A1L3X0,ELOVL7,Very long chain fatty acid elongase 7,6Y7F,A,A,37X,Octyl Glucose Neopentyl Glycol,CCCCCCC(CCCCCC)(CO[C@@H]1[C@H]([C@@H]([C@H]([C...,0.213523,C27H52O12,568.345877,0.099222,27.0,12.0,3.0,X-RAY DIFFRACTION,2.052
17,A0FGR8,ESYT2,Extended synaptotagmin-2,4P42,A,A,EGC,"2-(2-{2-[2-(2-{2-[2-(2-{2-[4-(1,1,3,3-TETRAMET...",CC(C)(C)CC(C)(C)c1ccc(cc1)OCCOCCOCCOCCOCCOCCOC...,1.000000,C32H58O10,602.402998,0.135862,32.0,10.0,1.0,X-RAY DIFFRACTION,2.440
82,A4D1P6,WDR91,WD repeat-containing protein 91,8SHJ,A,A,ZI8,N-[3-(4-chlorophenyl)oxetan-3-yl]-4-[(3S)-3-hy...,c1cc(ccc1C(=O)NC2(COC2)c3ccc(cc3)Cl)N4CC[C@@H]...,1.000000,C20H21ClN2O3,372.124070,0.865528,20.0,5.0,1.0,X-RAY DIFFRACTION,2.210
83,A4D1P6,WDR91,WD repeat-containing protein 91,8T55,C,C,ZI3,N-[3-(4-chlorophenyl)oxetan-3-yl]-1-propanoyl-...,CCC(=O)N1CCCc2c1cccc2C(=O)NC3(COC3)c4ccc(cc4)Cl,1.000000,C22H23ClN2O3,398.139720,0.854092,22.0,5.0,1.0,X-RAY DIFFRACTION,2.200


___
### Filter to drug-like fragements

In [11]:
df_pdb_ligands = pd.read_csv(ligand_out_csv)

# Filter to rows with a non empty ligand_code
df_ligands_filtered = df_pdb_ligands[df_pdb_ligands["ligand_code"].notna()]

# Filter dynamically based on filtering criteria
filtering_criteria = {
    "percent_intracellular": (0.8, 1.0),
    "mw": (100, 600),
    "qed": (0.44, 1.0),
    "num_N_O": (2, 100),
    "uniprot_id_count": (1, 2)
}

# Apply dynamic filtering
for col, (min_val, max_val) in filtering_criteria.items():
    # Drop rows where the filtering column is missing
    df_ligands_filtered = df_ligands_filtered[df_ligands_filtered[col].notna()]
    if min_val is not None:
        df_ligands_filtered = df_ligands_filtered[df_ligands_filtered[col] >= min_val]
    if max_val is not None:
        df_ligands_filtered = df_ligands_filtered[df_ligands_filtered[col] <= max_val]

print(f"df_ligands_filtered.shape: {df_ligands_filtered.shape}")
df_ligands_filtered.to_csv(frag_out_csv, index=False)
df_ligands_filtered.head()


df_ligands_filtered.shape: (13065, 18)


,uniprot_id,gene_name,recommendedName,pdb_id,protein_chain,ligand_chain,ligand_code,ligand_name,smiles,percent_intracellular,formula,mw,qed,num_carbon,num_N_O,uniprot_id_count,method,resolution
3,A4D1P6,WDR91,WD repeat-containing protein 91,8SHJ,A,A,ZI8,N-[3-(4-chlorophenyl)oxetan-3-yl]-4-[(3S)-3-hy...,c1cc(ccc1C(=O)NC2(COC2)c3ccc(cc3)Cl)N4CC[C@@H]...,1.0,C20H21ClN2O3,372.124070,0.865528,20.0,5.0,1.0,X-RAY DIFFRACTION,2.21
4,A4D1P6,WDR91,WD repeat-containing protein 91,8T55,C,C,ZI3,N-[3-(4-chlorophenyl)oxetan-3-yl]-1-propanoyl-...,CCC(=O)N1CCCc2c1cccc2C(=O)NC3(COC3)c4ccc(cc4)Cl,1.0,C22H23ClN2O3,398.139720,0.854092,22.0,5.0,1.0,X-RAY DIFFRACTION,2.20
5,A4D1P6,WDR91,WD repeat-containing protein 91,9DTA,A,A,A1BBV,"N-(1-benzoyl-1,2,3,4-tetrahydroquinolin-6-yl)-...",COc1c(cccc1C#N)CC(=O)Nc2ccc3c(c2)CCCN3C(=O)c4c...,1.0,C26H23N3O3,425.173942,0.659202,26.0,6.0,1.0,X-RAY DIFFRACTION,2.10
6,A4D1P6,WDR91,WD repeat-containing protein 91,9DTB,A,A,A1BBW,"N-[(1R)-4-cyano-2,3-dihydro-1H-inden-1-yl]-4-[...",c1ccc2c(c1)c(ncn2)Nc3ccc(cc3)C(=O)N[C@@H]4CCc5...,1.0,C25H19N5O,405.158960,0.515180,25.0,6.0,1.0,X-RAY DIFFRACTION,2.31
7,A4D1P6,WDR91,WD repeat-containing protein 91,9EJO,A,A,A1BIV,6-(4-chlorophenyl)-1-methyl-5-[(3S)-1-methyl-2...,CN1c2ccccc2[C@H](C1=O)c3c4c([nH]c3c5ccc(cc5)Cl...,1.0,C22H17ClN4O3,420.098918,0.521958,22.0,7.0,1.0,X-RAY DIFFRACTION,2.40


___
### Build a uniprot_gene_name_mapping.csv using uniprot_api_cache

In [ ]:
import json

# Read uniprot_ids from input_txt file
with open(input_txt, "r") as f:
    uniprot_ids = [line.strip() for line in f if line.strip()]

rows = []
for uniprot_id in uniprot_ids:
    uniprot_json_path = uniprot_cache_dir / f"{uniprot_id}.json"
    gene_name = recommended_name = None

    if uniprot_json_path.is_file():
        with open(uniprot_json_path, "r", encoding="utf-8") as f:
            envelope = json.load(f)

        if envelope.get("status_code") == 200:
            data = envelope.get("body") or {}

            genes = data.get("genes") or []
            if genes:
                gene_name = (genes[0].get("geneName") or {}).get("value")

            protein_desc = data.get("proteinDescription") or {}
            rec = protein_desc.get("recommendedName") or {}
            recommended_name = (rec.get("fullName") or {}).get("value")
            if recommended_name is None:
                submission_names = protein_desc.get("submissionNames") or []
                if submission_names:
                    recommended_name = (submission_names[0].get("fullName") or {}).get("value")

    rows.append(
        {
            "uniprot_id": uniprot_id,
            "gene_name": gene_name,
            "recommendedName": recommended_name,
        }
    )

df = pd.DataFrame(rows, columns=["uniprot_id", "gene_name", "recommendedName"])
df.head()
